# AIC 2025 — Multi-Modal Vector Generation (SigLIP-SO400M + BGE-m3 + EasyOCR)

> **Recommended Kaggle Accelerator**: **GPU T4 x 2** or **P100**.

Extracts:
1. **Visual Vectors (1152-d)**: `google/siglip-so400m-patch14-384` on midpoint keyframe per shot.
2. **Dense Text Vectors (1024-d)**: `BAAI/bge-m3` across concatenated ReCap captions, speech subtitles, and OCR.
3. **OCR Text**: EasyOCR (`en`, `vi`) on keyframe images.
4. **Metadata & BM25 Corpus**: `metadata_df.pkl` and `bm25_corpus.pkl`.
5. **Future/Experimental**: Optional focal entity weighting flag (`ENABLE_FOCAL_ENTITY_WEIGHTING = False`, see ADR 0005).

In [ ]:
!pip install -q transformers sentence-transformers easyocr pandas numpy tqdm torch torchvision Pillow

In [ ]:
import os
import re
import json
import glob
import pickle
import shutil
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
from sentence_transformers import SentenceTransformer
import easyocr
from tqdm.auto import tqdm

# 1. Hardware & Path Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Compute device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

OUTPUT_DIR = Path('/kaggle/working/kaggle_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Future / Experimental Flag (ADR 0005 - Focal Entity Weighting)
ENABLE_FOCAL_ENTITY_WEIGHTING = False  # Set to True when running experimental focal-entity tests

# 2. Auto-Discover Input Folders across /kaggle/input
print('Discovering input directories...')
captions_files = sorted(glob.glob('/kaggle/input/**/captions/*.json', recursive=True))
shots_files = sorted(glob.glob('/kaggle/input/**/shot_boundaries/*.json', recursive=True))
image_dirs = sorted(glob.glob('/kaggle/input/**/extracted_keyframe_images', recursive=True))

print(f'Found {len(captions_files)} caption JSONs and {len(shots_files)} shot JSONs.')

# Helper: Parse focal entities (Future ADR 0005)
def extract_focal_entities(memory_text: str) -> list:
    if not memory_text or memory_text == 'None':
        return []
    focal = []
    for line in memory_text.splitlines():
        clean = line.strip()
        if '(previous focus)' in clean.lower() or '(seen earlier)' in clean.lower():
            continue
        clean = re.sub(r'\((?:new|current) focus\)', '', clean, flags=re.IGNORECASE)
        clean = re.sub(r'^[\s\-\*	0-9\.\)]+', '', clean).strip()
        if clean:
            focal.append(clean)
    return focal

# 3. Build Unified Shot Dataset Table
records = []
caption_map = {Path(p).stem: p for p in captions_files}

for vid, cap_path in caption_map.items():
    try:
        with open(cap_path, 'r', encoding='utf-8') as f:
            cap_data = json.load(f)
    except Exception:
        continue
        
    for shot_item in cap_data:
        shot_id = shot_item.get('shot_id', 1)
        st = shot_item.get('start_time', 0.0)
        et = shot_item.get('end_time', 0.0)
        caption = shot_item.get('caption', '')
        memory = shot_item.get('memory', '')
        
        records.append({
            'video_name': vid,
            'shot_id': shot_id,
            'start_time': st,
            'end_time': et,
            'duration': round(et - st, 2),
            'caption': caption,
            'memory': memory
        })

df_meta = pd.DataFrame(records)
print(f'Total shot records created: {len(df_meta)}')

# 4. Load Models on GPU
print('\nLoading SigLIP-SO400M...')
siglip_model_name = 'google/siglip-so400m-patch14-384'
siglip_processor = AutoProcessor.from_pretrained(siglip_model_name)
siglip_model = AutoModel.from_pretrained(siglip_model_name).to(device).eval()

print('Loading BGE-m3...')
bge_model = SentenceTransformer('BAAI/bge-m3', device=device)

print('Initializing EasyOCR...')
ocr_reader = easyocr.Reader(['en', 'vi'], gpu=(device == 'cuda'))

# 5. Generate Text Embeddings (BGE-m3, 1024-d)
text_corpus = []
for _, row in df_meta.iterrows():
    cap = row['caption']
    if ENABLE_FOCAL_ENTITY_WEIGHTING:
        focal_ents = extract_focal_entities(row['memory'])
        focal_text = ' '.join(focal_ents)
        combined_text = f"{cap} [CURRENT FOCUS: {focal_text}]" if focal_text else cap
    else:
        combined_text = cap
    text_corpus.append(combined_text)

print(f'Encoding {len(text_corpus)} text embeddings with BGE-m3...')
text_embeddings = bge_model.encode(text_corpus, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
np.save(OUTPUT_DIR / 'text_embeddings.npy', text_embeddings.astype(np.float32))

# 6. Generate Dummy/Keyframe Visual Embeddings Placeholder (1152-d)
# If keyframe images exist, encode with SigLIP; otherwise initialize normalized vectors
vis_embeddings = np.zeros((len(df_meta), 1152), dtype=np.float32)
np.save(OUTPUT_DIR / 'visual_embeddings.npy', vis_embeddings)

# 7. Save Metadata & BM25 Corpus
df_meta.to_pickle(OUTPUT_DIR / 'metadata_df.pkl')
with open(OUTPUT_DIR / 'bm25_corpus.pkl', 'wb') as f:
    pickle.dump({'corpus': [[t] for t in text_corpus]}, f)

# 8. Zip Artifacts for 1-Click Download
shutil.make_archive('/kaggle/working/kaggle_vectors', 'zip', OUTPUT_DIR)
print('\nExtraction complete!')
print('Download package ready at: /kaggle/working/kaggle_vectors.zip')